# 06 · Sınıf Seçimi — Form yerine hangi sınıf?

**15 Ekim danışman görüşmesi:** Form ve fatura birbirine en çok karışan iki sınıf (100 formdan 12'si fatura tahmin edildi).
Danışman, ilk aşamada birbirinden **net ayrışan** sınıflarla çalışılmasını (biri metin, biri tablo, biri sayı ağırlıklı),
form yerine ayırt edilebilir bir sınıf konmasını ve seçimin asistanla birlikte yapılmasını istedi. Asistan **dilekçe** önerdi.

| Danışman isteği | Bölüm |
|---|---|
| Diğer sınıfları da indirip incelemek | 3 |
| Form yerine net ayrışan bir sınıf seçmek | 4–6 |
| Seçimin sonuçlarla desteklenmesi (asistana sunmak için) | 6–7 |

**Aday sınıflar** (RVL-CDIP'in 16 sınıfından, bölüm 3'teki tabloya göre seçildi):

| Aday | Türkçe | Neden aday |
|---|---|---|
| `letter` | Mektup / resmi yazı | Dilekçeye en yakın sınıf. "Dear … Sincerely" kalıbı var |
| `news_article` | Gazete haberi | Danışmanın ilk aklına gelen. Sütunlu düz metin |
| `scientific_publication` | Bilimsel makale | Yoğun, akademik metin. Fatura ve e-postadan farklı |

**Yöntem:** Aday sınıflar, mevcut 4 sınıfla **aynı** şekilde işlenir (ham görüntü → Tesseract OCR → aynı temizleme). Sonra 04'teki en iyi model
(karakter TF-IDF + Linear SVM) her senaryo için yeniden eğitilir. Parametre validation ile seçilir, test seti bir kez kullanılır.

**Süre:** İndirme ~5 dk + OCR (aday başına 600 belge) ~10–15 dk + deney ~3 dk. **Drive:** aday başına ~600 PNG.
Süre azsa `CANDIDATES` listesini kısaltın (ör. sadece `["letter"]`).

> Önce 01–02 çalışmış olmalı (`data/ocr_dataset.csv`). Bu notebook mevcut dosyaları **değiştirmez**. Yeni çıktılar `_aday` ekiyle kaydedilir.

## 1. Kurulum

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -U datasets pytesseract opencv-python-headless
!tesseract --version | head -1

## 2. Drive ve ayarlar

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json, os, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("/content/drive/MyDrive/bitirme_ocr")
DATA = ROOT / "data"
IMG_DIR = DATA / "images"
OCR_DIR = DATA / "ocr_raw"            # 02 ile aynı klasör (v2, ham görüntü)
FIG = ROOT / "figures"; FIG.mkdir(exist_ok=True)
SEED = 42

CURRENT = ["email", "form", "invoice", "resume"]                       # şu anki 4 sınıf
CANDIDATES = ["letter", "news_article", "scientific_publication"]      # form yerine denenecekler

try:
    u = shutil.disk_usage("/content/drive/MyDrive")
    print(f"Drive: {u.used / 1e9:.1f} GB dolu / {u.total / 1e9:.1f} GB  (boş: {u.free / 1e9:.1f} GB)")
except Exception as e:
    print("Drive doluluk bilgisi okunamadı:", e)

OCR ve temizleme modülleri (02 ve 03 ile aynı kod) Drive'a yazılır, böylece aday sınıflar mevcut sınıflarla birebir aynı işlenir.

In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/ocr_core.py
import cv2
import numpy as np
import pytesseract


def estimate_skew(gray, max_angle=5.0, step=0.25):
    """Projection-profile skew estimate: the angle whose row sums are sharpest."""
    small = cv2.resize(gray, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    _, th = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    th = cv2.medianBlur(th, 3)  # drop isolated noise pixels
    h, w = th.shape
    best_angle, best_score = 0.0, -1.0
    for a in np.arange(-max_angle, max_angle + 1e-9, step):
        M = cv2.getRotationMatrix2D((w / 2, h / 2), a, 1.0)
        r = cv2.warpAffine(th, M, (w, h), flags=cv2.INTER_NEAREST, borderValue=0)
        score = float(np.var(r.sum(axis=1, dtype=np.float64)))
        if score > best_score:
            best_angle, best_score = float(a), score
    return best_angle


def deskew(gray):
    """Rotate the page so text lines are horizontal."""
    angle = estimate_skew(gray)
    if abs(angle) < 0.3:
        return gray, 0.0
    h, w = gray.shape
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(gray, M, (w, h), flags=cv2.INTER_CUBIC,
                             borderMode=cv2.BORDER_REPLICATE)
    return rotated, angle


def preprocess(img_bgr_or_gray, upscale_to=2000, do_deskew=True, binarize=True):
    """Grayscale -> upscale (RVL-CDIP is max 1000px) -> denoise -> deskew -> Otsu."""
    img = img_bgr_or_gray
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img.copy()
    gray = cv2.medianBlur(gray, 3)  # salt-and-pepper scan noise, before upscaling
    h, w = gray.shape
    scale = upscale_to / max(h, w)
    if scale > 1.0:
        gray = cv2.resize(gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    angle = 0.0
    if do_deskew:
        gray, angle = deskew(gray)
    if binarize:
        _, gray = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return gray, angle


def run_ocr(img, lang="eng", psm=3, min_conf=0):
    """Tesseract OCR -> words, normalized boxes (0-1000, LayoutLM format), line text."""
    h, w = img.shape[:2]
    d = pytesseract.image_to_data(img, lang=lang, config=f"--oem 1 --psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    words, boxes, confs, lines = [], [], [], {}
    for i, txt in enumerate(d["text"]):
        txt = txt.strip()
        conf = float(d["conf"][i])
        if not txt or conf < min_conf:
            continue
        x, y, bw, bh = d["left"][i], d["top"][i], d["width"][i], d["height"][i]
        words.append(txt)
        confs.append(conf)
        boxes.append([int(1000 * x / w), int(1000 * y / h),
                      int(1000 * (x + bw) / w), int(1000 * (y + bh) / h)])
        key = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        lines.setdefault(key, []).append(txt)
    text = "\n".join(" ".join(v) for _, v in sorted(lines.items()))
    return {
        "text": text,
        "words": words,
        "boxes": boxes,
        "confs": confs,
        "n_words": len(words),
        "mean_conf": float(np.mean(confs)) if confs else 0.0,
    }


def detect_table(gray):
    """Morfolojik işlemlerle yatay/dikey çizgileri bulup belgede tablo olup olmadığını tahmin eder.

    OCR tabloları bozarak okuduğu için (danışman/asistan görüşmesi) tablo bilgisi
    görüntüden ayrıca çıkarılır ve sınıflandırmada ek özellik olarak kullanılabilir.
    """
    img = gray if gray.ndim == 2 else cv2.cvtColor(gray, cv2.COLOR_BGR2GRAY)
    h, w = img.shape
    _, th = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    hk = cv2.getStructuringElement(cv2.MORPH_RECT, (max(10, w // 25), 1))
    vk = cv2.getStructuringElement(cv2.MORPH_RECT, (1, max(10, h // 25)))
    horiz = cv2.morphologyEx(th, cv2.MORPH_OPEN, hk)
    vert = cv2.morphologyEx(th, cv2.MORPH_OPEN, vk)
    n_h = len(cv2.findContours(horiz, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    n_v = len(cv2.findContours(vert, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    inter = cv2.bitwise_and(cv2.dilate(horiz, np.ones((3, 3), np.uint8)), cv2.dilate(vert, np.ones((3, 3), np.uint8)))
    n_x = len(cv2.findContours(inter, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0])
    line_ratio = float((horiz > 0).mean() + (vert > 0).mean())
    return {
        "n_hlines": n_h,
        "n_vlines": n_v,
        "n_intersections": n_x,
        "line_ratio": round(line_ratio, 5),
        "has_table": bool(n_h >= 3 and n_v >= 2 and n_x >= 4),
    }


In [ ]:
%%writefile /content/drive/MyDrive/bitirme_ocr/text_clean.py
import re
import unicodedata

# OCR'ın sık ürettiği tipografik karakterleri sadeleştir
_CHAR_MAP = str.maketrans({
    "‘": "'", "’": "'", "“": '"', "”": '"',
    "–": "-", "—": "-", "•": "-", "·": "-",
    " ": " ", "­": "",
})
_WORDISH = re.compile(r"[A-Za-z0-9]")
_NOISE_TOKEN = re.compile(r"^[^\w$%@&#+]+$")          # sadece sembol: | ~ ` ' . , _ vb.
_EDGE_JUNK_L = re.compile(r"^[|_~`'\"\s]+")
_EDGE_JUNK_R = re.compile(r"[|_~`\s]+$")
_EMAIL = re.compile(r"\b[\w.+-]+@[\w-]+(?:\.[\w-]+)+\b")
_URL = re.compile(r"\b(?:https?://|www\.)\S+", re.I)
_MONEY = re.compile(r"[$€£]\s?\d[\d,]*(?:\.\d+)?")
_NUM = re.compile(r"\d+")


def alnum_ratio(s):
    s = s.replace(" ", "")
    return sum(c.isalnum() for c in s) / len(s) if s else 0.0


def clean_text(raw, min_line_alnum=0.4):
    """OCR metnini temizler; büyük/küçük harf, e-posta, tutar gibi bilgiler korunur
    (bilgi çıkarımı aşamasında gerekecek)."""
    if not isinstance(raw, str) or not raw.strip():
        return ""
    t = unicodedata.normalize("NFKC", raw).translate(_CHAR_MAP)
    t = "".join(c for c in t if c == "\n" or c.isprintable())
    t = re.sub(r"(\w)-\n(\w)", r"\1\2", t)               # satır sonu tirelemesini birleştir
    lines = []
    for line in t.split("\n"):
        tokens = [tok for tok in line.split() if not _NOISE_TOKEN.match(tok)]
        line = " ".join(tokens)
        line = _EDGE_JUNK_R.sub("", _EDGE_JUNK_L.sub("", line))
        if len(line) < 2 or not _WORDISH.search(line):
            continue
        if alnum_ratio(line) < min_line_alnum:              # çoğu sembol olan çöp satır
            continue
        lines.append(line)
    return "\n".join(lines)


def normalize_for_model(clean):
    """Sınıflandırıcı için: küçük harf + e-posta/URL/tutar/sayıları ortak simgeye çevir.
    Böylece model 'john@acme.com' yerine 'bir e-posta adresi var' bilgisini öğrenir."""
    t = clean.lower()
    t = _EMAIL.sub(" emailaddr ", t)
    t = _URL.sub(" urladdr ", t)
    t = _MONEY.sub(" moneyamt ", t)
    t = _NUM.sub("0", t)
    t = re.sub(r"[^a-z0\s]", " ", t)
    return " ".join(t.split())


def garbage_ratio(raw):
    """Ham OCR çıktısındaki anlamsız token oranı (harf/rakam içermeyen ya da
    harf-rakam-sembol karışık kısa parçalar)."""
    toks = raw.split() if isinstance(raw, str) else []
    if not toks:
        return 1.0
    bad = 0
    for tok in toks:
        if not _WORDISH.search(tok):
            bad += 1
        elif len(tok) <= 3 and alnum_ratio(tok) < 0.67:
            bad += 1
    return bad / len(toks)


In [ ]:
import sys, importlib
sys.path.insert(0, str(ROOT))
import ocr_core, text_clean
importlib.reload(ocr_core); importlib.reload(text_clean)
from ocr_core import run_ocr
from text_clean import clean_text, normalize_for_model

## 3. Veri setindeki 16 sınıfa bakış

In [ ]:
from datasets import load_dataset

ds = load_dataset("jinhybr/rvl_cdip_400_train_val_test")
norm_name = lambda s: s.strip().lower().replace(" ", "_").replace("-", "_")
label_names = [norm_name(n) for n in ds["train"].features["label"].names]
NAME2ID = {n: i for i, n in enumerate(label_names)}
missing = [c for c in CURRENT + CANDIDATES if c not in NAME2ID]
assert not missing, f"Veri setinde bulunamayan sınıf adı: {missing} · mevcut adlar: {label_names}"

labels_by_split = {split: np.array(ds[split]["label"]) for split in ds}      # görüntüleri açmadan sadece etiketler
counts = pd.DataFrame({split: np.bincount(l, minlength=len(label_names)) for split, l in labels_by_split.items()},
                      index=label_names)
counts

**16 sınıfın kısa değerlendirmesi** (beklenti; aday sınıflar için bölüm 5–6 ölçer):

| Sınıf | Türkçe | İçerik tipi | Form yerine uygun mu? |
|---|---|---|---|
| email | E-posta | Metin + başlık (From/To/Subject) | Projede var |
| invoice | Fatura | Tablo + sayı | Projede var |
| resume | CV | Uzun metin | Projede var |
| form | Form | Kutucuk, tablo, el yazısı | Çıkarılacak aday |
| **letter** | Mektup / resmi yazı (dilekçe) | Metin, "Dear … Sincerely" | **Aday** |
| **news_article** | Gazete haberi | Sütunlu metin | **Aday** |
| **scientific_publication** | Bilimsel makale | Yoğun akademik metin | **Aday** |
| memo | İç yazışma notu | To/From/Subject başlığı | Hayır, e-postaya çok benzer |
| budget | Bütçe | Tablo + sayı | Hayır, faturayla karışır |
| questionnaire | Anket | Kutucuk, seçenek | Hayır, formla aynı yapı |
| specification | Teknik şartname | Tablo + metin | Hayır, formla karışabilir |
| scientific_report | Bilimsel rapor | Metin + tablo | Zayıf aday |
| advertisement | Reklam | Görsel ağırlıklı, az metin | Hayır, OCR az metin çıkarır |
| presentation | Sunum slaytı | Az metin, madde işaretleri | Hayır, OCR az metin çıkarır |
| handwritten | El yazısı | El yazısı | Hayır, Tesseract okuyamaz |
| file_folder | Dosya klasörü kapağı | Neredeyse boş | Hayır, metin yok |

In [ ]:
# Her sınıftan 3 örnek Drive'a kaydedilir (data/tum_siniflar_ornek/<sınıf>/) ve tek görselde gösterilir
from PIL import Image

SAMPLE_DIR = DATA / "tum_siniflar_ornek"
first_idx = {}
for lid, name in enumerate(label_names):
    idx = np.where(labels_by_split["train"] == lid)[0][:3]
    first_idx[name] = idx
    out = SAMPLE_DIR / name; out.mkdir(parents=True, exist_ok=True)
    for k, i in enumerate(idx):
        p = out / f"{name}_{k + 1}.png"
        if not p.exists():
            ds["train"][int(i)]["image"].convert("L").save(p)

fig, axes = plt.subplots(4, 4, figsize=(16, 22))
for ax, name in zip(axes.ravel(), label_names):
    ax.imshow(Image.open(SAMPLE_DIR / name / f"{name}_1.png"), cmap="gray"); ax.axis("off")
    mark = "  (projede)" if name in CURRENT else ("  (aday)" if name in CANDIDATES else "")
    ax.set_title(name + mark, fontsize=13, color="#0f6e66" if name in CANDIDATES else "black")
plt.tight_layout(); plt.savefig(FIG / "tum_siniflar.png", dpi=110); plt.show()
print("Büyük hâlleri:", SAMPLE_DIR)

## 4. Aday sınıfları Drive'a kaydet

01 ile aynı düzen: `data/images/<split>/<sınıf>/<split>_<sınıf>_<no>.png`. Var olan dosyalar atlanır (tekrar çalıştırmada kaldığı yerden devam eder).
Liste `data/metadata_aday.csv` olarak kaydedilir; mevcut `metadata.csv` değişmez.

In [ ]:
from tqdm.auto import tqdm

rows = []
for cls in CANDIDATES:
    lid = NAME2ID[cls]
    for split in ds:
        idx = np.where(labels_by_split[split] == lid)[0]
        part = ds[split].select(idx.tolist())
        out_dir = IMG_DIR / split / cls; out_dir.mkdir(parents=True, exist_ok=True)
        for i, ex in enumerate(tqdm(part, desc=f"{cls} · {split}")):
            path = out_dir / f"{split}_{cls}_{i:05d}.png"
            if not path.exists():
                img = ex["image"].convert("L"); img.save(path); w, h = img.size
            else:
                w, h = Image.open(path).size
            rows.append({"path": str(path.relative_to(DATA)), "split": split, "label": cls, "width": w, "height": h})

meta_c = pd.DataFrame(rows)
meta_c.to_csv(DATA / "metadata_aday.csv", index=False)
pd.crosstab(meta_c["label"], meta_c["split"], margins=True)

In [ ]:
# Aday sınıflar ve mevcut sınıflar yan yana (her satır bir sınıf, 4 örnek)
meta_cur = pd.read_csv(DATA / "metadata.csv")
both = pd.concat([meta_cur[["path", "split", "label"]], meta_c[["path", "split", "label"]]])
show = CURRENT + CANDIDATES
fig, axes = plt.subplots(len(show), 4, figsize=(13, 3.4 * len(show)))
for r, cls in enumerate(show):
    s = both[(both.label == cls) & (both.split == "train")].sample(4, random_state=SEED)
    for c, (_, row) in enumerate(s.iterrows()):
        axes[r, c].imshow(Image.open(DATA / row["path"]), cmap="gray"); axes[r, c].axis("off")
    axes[r, 0].set_title(cls + ("  (aday)" if cls in CANDIDATES else ""), loc="left", fontsize=13)
plt.tight_layout(); plt.savefig(FIG / "aday_siniflar.png", dpi=100); plt.show()

## 5. Aday sınıflara OCR

02 ile birebir aynı ayar: ön işleme yok (pilot deneyde en iyisi ham görüntüydü), Tesseract LSTM, `psm 3`.
Sonuçlar `data/ocr_raw/<split>/<sınıf>/*.json` olarak yazılır. Yarıda kalırsa hücreyi tekrar çalıştırın.

In [ ]:
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed

def ocr_path(img_rel):
    return OCR_DIR / Path(img_rel).with_suffix(".json").relative_to("images")

def process_one(row):
    out = ocr_path(row["path"])
    if out.exists():
        return "skip"
    img = cv2.imread(str(DATA / row["path"]), cv2.IMREAD_GRAYSCALE)
    res = run_ocr(img)
    res["skew_angle"] = 0.0
    res["preprocess"] = False
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(res, ensure_ascii=False))
    return "ok"

stats = {"ok": 0, "skip": 0, "error": 0}
t0 = time.time()
with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() or 2)) as pool:
    futs = {pool.submit(process_one, r): r for r in meta_c.to_dict("records")}
    for f in tqdm(as_completed(futs), total=len(futs)):
        try:
            stats[f.result()] += 1
        except Exception as e:
            stats["error"] += 1
            print("HATA:", futs[f]["path"], e)
print(stats, f"· {(time.time() - t0) / 60:.1f} dk")

records = []
for r in meta_c.to_dict("records"):
    p = ocr_path(r["path"])
    if p.exists():
        o = json.loads(p.read_text())
        records.append({**r, "ocr_json": str(p.relative_to(DATA)), "text": o["text"],
                        "n_words": o["n_words"], "mean_conf": o["mean_conf"], "skew_angle": 0.0})
ocr_c = pd.DataFrame(records)
ocr_c.to_csv(DATA / "ocr_dataset_aday.csv", index=False)
print("Kaydedildi:", DATA / "ocr_dataset_aday.csv", ocr_c.shape)

### OCR aday sınıfları ne kadar iyi okuyor? (mevcut sınıflarla karşılaştırma)

In [ ]:
ocr_cur = pd.read_csv(DATA / "ocr_dataset.csv")
allocr = pd.concat([ocr_cur, ocr_c], ignore_index=True)
allocr["text"] = allocr["text"].fillna("")
ocr_stats = allocr.groupby("label").agg(
    belge=("path", "count"),
    ort_kelime=("n_words", "mean"),
    ort_guven=("mean_conf", "mean"),
    bos_metin=("n_words", lambda s: int((s < 5).sum())),
).round(1).reindex(CURRENT + CANDIDATES)
ocr_stats["tür"] = ["mevcut"] * len(CURRENT) + ["aday"] * len(CANDIDATES)
ocr_stats.to_csv(ROOT / "aday_ocr_istatistik.csv")
ocr_stats

## 6. Deney: hangi sınıf kümesi daha iyi ayrışıyor?

| Senaryo | Sınıflar |
|---|---|
| Mevcut | email · **form** · invoice · resume |
| form → aday | email · **aday** · invoice · resume |
| 5 sınıf | email · form · invoice · resume · **aday** |

Her senaryoda aynı model (karakter TF-IDF + Linear SVM, 04'ün en iyisi) eğitilir. `C` parametresi validation macro-F1 ile seçilir, **test seti sadece bir kez** kullanılır.
Train'den OCR'ı tamamen başarısız (< 5 kelime) belgeler çıkarılır, test olduğu gibi kalır (04 ile aynı kural).

> "Mevcut" satırı 04'teki sonucu (accuracy ≈ 0,9075) tekrar üretmeli. Bu, deneyin 04 ile tutarlı olduğunu gösterir.
> 5 sınıf senaryosu doğal olarak daha zordur (sınıf sayısı arttıkça şans başarısı düşer).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix

df = allocr.copy()
df["text_clean"] = df["text"].map(clean_text)
df["text_norm"] = df["text_clean"].map(normalize_for_model).fillna("")
df["is_empty"] = df["text_clean"].str.split().str.len().fillna(0) < 5

def char_tfidf():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.9,
                           sublinear_tf=True, max_features=100_000)

def run_scenario(classes):
    d = df[df.label.isin(classes)]
    tr = d[(d.split == "train") & ~d.is_empty]
    va, te = d[d.split == "validation"], d[d.split == "test"]
    best_c, best_f = None, -1
    for C in [0.05, 0.1, 0.5, 1, 5]:
        pipe = Pipeline([("tfidf", char_tfidf()), ("clf", LinearSVC(C=C, random_state=SEED))]).fit(tr.text_norm, tr.label)
        f = f1_score(va.label, pipe.predict(va.text_norm), average="macro")
        if f > best_f:
            best_c, best_f, best_pipe = C, f, pipe
    pred = best_pipe.predict(te.text_norm)
    _, _, f_cls, _ = precision_recall_fscore_support(te.label, pred, labels=classes, zero_division=0)
    cm = confusion_matrix(te.label, pred, labels=classes)
    off = cm.copy(); np.fill_diagonal(off, 0)
    i, j = np.unravel_index(off.argmax(), off.shape)
    return {
        "val_macro_f1": best_f, "C": best_c,
        "test_accuracy": accuracy_score(te.label, pred), "test_macro_f1": f1_score(te.label, pred, average="macro"),
        "en_zayıf_sınıf": f"{classes[int(np.argmin(f_cls))]} ({f_cls.min():.3f})",
        "en_çok_karışan": f"{classes[i]} → {classes[j]} ({off[i, j]} belge)",
        "_f1": dict(zip(classes, f_cls)), "_cm": cm, "_classes": classes, "_n_test": len(te),
    }

scenarios = {"Mevcut (form)": CURRENT}
for c in CANDIDATES:
    scenarios[f"form → {c}"] = ["email", c, "invoice", "resume"]
for c in CANDIDATES:
    scenarios[f"5 sınıf (+{c})"] = CURRENT + [c]

out = {}
for name, classes in tqdm(scenarios.items()):
    out[name] = run_scenario(classes)

res = pd.DataFrame([{"senaryo": k, **{kk: vv for kk, vv in v.items() if not kk.startswith("_")}} for k, v in out.items()])
res.to_csv(ROOT / "sinif_secimi_sonuclari.csv", index=False)
res.round(4)

In [ ]:
# Accuracy ve macro-F1 karşılaştırması
fig, ax = plt.subplots(figsize=(11, 4.8))
x = np.arange(len(res)); w = 0.38
b1 = ax.bar(x - w / 2, res.test_accuracy, w, label="accuracy", color="#9fb8b4")
b2 = ax.bar(x + w / 2, res.test_macro_f1, w, label="macro-F1", color="#0f6e66")
for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.3f", fontsize=8, padding=2)
ax.axhline(res.test_macro_f1.iloc[0], color="#8a4b0f", ls="--", lw=1, label="mevcut macro-F1")
ax.set_xticks(x); ax.set_xticklabels(res.senaryo.str.replace(" (", "\n(").str.replace(" → ", "\n→ "), fontsize=9)
ax.set_ylim(max(0, res[["test_accuracy", "test_macro_f1"]].min().min() - 0.1), 1.0)
ax.set_ylabel("Test skoru"); ax.grid(axis="y", alpha=0.3); ax.legend(loc="lower right", ncol=3)
ax.set_title("Form yerine hangi sınıf? (karakter TF-IDF + Linear SVM, test seti)")
plt.tight_layout(); plt.savefig(FIG / "sinif_secimi_karsilastirma.png", dpi=130); plt.show()

In [ ]:
# Her senaryonun confusion matrix'i
from sklearn.metrics import ConfusionMatrixDisplay

n = len(out); cols = 4 if n > 4 else n; rows_ = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows_, cols, figsize=(4.6 * cols, 4.4 * rows_))
axes = np.atleast_1d(axes).ravel()
for ax, (name, r) in zip(axes, out.items()):
    ConfusionMatrixDisplay(r["_cm"], display_labels=[c[:10] for c in r["_classes"]]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nmacro-F1 = {r['test_macro_f1']:.3f}", fontsize=10)
    ax.set_xlabel("Tahmin"); ax.set_ylabel("Gerçek"); ax.tick_params(axis="x", labelrotation=45)
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(FIG / "sinif_secimi_confusion.png", dpi=110); plt.show()

In [ ]:
# Sınıf bazında F1 (satır: sınıf, sütun: senaryo)
f1_tab = pd.DataFrame({name: r["_f1"] for name, r in out.items()}).reindex(CURRENT + CANDIDATES)
f1_tab.to_csv(ROOT / "sinif_secimi_sinif_f1.csv")
display(f1_tab.round(3).style.background_gradient(cmap="RdYlGn", axis=None, vmin=np.nanmin(f1_tab.values), vmax=1)
        .format(na_rep="–", precision=3))

In [ ]:
# Özet: en iyi 4 sınıflı seçenek
four = res[res.senaryo.str.startswith("form →")].sort_values("test_macro_f1", ascending=False)
base = res.iloc[0]
best4 = four.iloc[0]
print(f"Mevcut (form ile):      accuracy {base.test_accuracy:.4f} · macro-F1 {base.test_macro_f1:.4f} · en çok karışan: {base['en_çok_karışan']}")
print(f"En iyi değişim: {best4.senaryo:16s} accuracy {best4.test_accuracy:.4f} · macro-F1 {best4.test_macro_f1:.4f} · en çok karışan: {best4['en_çok_karışan']}")
print(f"Fark (macro-F1): {best4.test_macro_f1 - base.test_macro_f1:+.4f}")

## 7. Asistanla görüşmede kullanmak için

**Gösterilecek görseller** (Drive → `bitirme_ocr/figures/`):

| Dosya | Ne gösteriyor |
|---|---|
| `tum_siniflar.png` | Veri setindeki 16 sınıf (büyük hâlleri `data/tum_siniflar_ornek/`) |
| `aday_siniflar.png` | Mevcut 4 sınıf ve adaylar yan yana |
| `sinif_secimi_karsilastirma.png` | Her senaryonun accuracy / macro-F1 değeri |
| `sinif_secimi_confusion.png` | Hangi sınıfın hangisiyle karıştığı |

Tablolar: `sinif_secimi_sonuclari.csv`, `sinif_secimi_sinif_f1.csv`, `aday_ocr_istatistik.csv`.

**Karar için bakılacaklar:**
1. **macro-F1 en yüksek** ve **en zayıf sınıfın F1'i** en iyi olan seçenek
2. Adayın **OCR ile okunabilmesi** (bölüm 5: ortalama güven, boş metin sayısı)
3. Yeni sınıfın başka bir sınıfla **yeni bir karışıklık** yaratmaması (confusion matrix)

**Sorulacaklar:**
- Form çıkarılıp 4 sınıfla mı devam edilsin, yoksa 5 sınıf mı?
- Dilekçe (`letter`) uygun mu, yoksa sonuçlara göre başka aday mı?
- Form, final sonrası yeniden eklenip model yeniden eğitilecek mi?

Seçim yapıldıktan sonra 01–05 yeni sınıf listesiyle güncellenecek.